## Imports

In [1]:
import pandas as pd
import plotly.express as px

## Dicionário de Dados: Consumo de Energia Residencial

Este documento descreve os campos e metadados contidos no conjunto de dados de consumo de energia.

A GoodWe não fabrica apenas inversores, ela fornece ecossistemas onde o usuário monitora para onde a energia solar está indo. O dataset detalha o "Appliance Type", o que permite à GoodWe aprimorar algoritmos que dizem ao cliente, "Sua geladeira está consumindo mais do que o normal".

### Atributos

| Atributo | Descrição |
| :--- | :--- |
| **Home ID** | Um identificador exclusivo para cada residência (anonimizado). |
| **Appliance Type** | O eletrodoméstico específico em uso (ex: Geladeira, Forno, Aquecedor). |
| **Energy Consumption (kWh)** | Energia consumida pelo aparelho em quilowatts-hora (kWh). |
| **Time** | O horário em que ocorreu o consumo de energia (formato 24 horas). |
| **Date** | A data em que o consumo de energia foi registrado (AAAA-MM-DD). |
| **Outdoor Temperature (°C)** | A temperatura externa no momento do consumo de energia, em graus Celsius. |
| **Season** | A estação do ano (Inverno, Verão, Outono, Primavera). |
| **Household Size** | Número de pessoas residentes na casa. |


In [2]:
df= pd.read_csv('./smart_home_energy_consumption_large.csv')

display(df)

,Home ID,Appliance Type,Energy Consumption (kWh),Time,Date,Outdoor Temperature (°C),Season,Household Size
0,94,Fridge,0.20,21:12,2023-12-02,-1.0,Fall,2
1,435,Oven,0.23,20:11,2023-08-06,31.1,Summer,5
2,466,Dishwasher,0.32,06:39,2023-11-21,21.3,Fall,3
3,496,Heater,3.92,21:56,2023-01-21,-4.2,Winter,1
4,137,Microwave,0.44,04:31,2023-08-26,34.5,Summer,5
...,...,...,...,...,...,...,...,...
99995,124,Microwave,0.42,09:56,2023-09-28,20.5,Summer,1
99996,184,Computer,0.71,12:48,2023-05-27,-5.4,Spring,2
99997,101,Dishwasher,0.25,05:45,2023-02-18,35.6,Winter,3
99998,423,Air Conditioning,2.69,12:39,2023-04-20,3.7,Spring,1


In [3]:
df_corr = df.select_dtypes(include="number")

corr = df_corr.corr()

px.imshow(
    corr,
    text_auto=True,
    aspect="auto",
    title="Matriz de Correlação"
)

## a) 1 variável quantitativa discreta, na sequência, extraia pelo menos 2 insights da tabela utilizando #: **Household Size**

### Esta variável é classificada como quantitativa discreta pois representa uma contagem de unidades inteiras (pessoas), onde não existem valores fracionários (não é possível ter 2,5 pessoas em uma casa).
### Insights Extraídos:

- Correlação Consumo vs. Densidade Populacional: 

Embora a variável Household Size permita investigar a relação entre número de moradores e consumo de energia, os resultados indicam que não há correlação linear significativa entre essas variáveis. O consumo total de energia apresenta pouca variação entre residências de diferentes tamanhos, sugerindo que fatores como tipo de aparelho e padrão de uso têm maior influência no consumo energético.

- Identificação de Outliers de Eficiência: 

Ao analisar o consumo por pessoa (kWh per capita), observa-se que residências com poucos moradores apresentam valores significativamente mais elevados em comparação com residências maiores. Esse comportamento indica a presença de outliers de eficiência, possivelmente associados ao uso intensivo de aparelhos de alto consumo, como aquecedores e sistemas de ar-condicionado, ou à ausência de compartilhamento do consumo energético.

Esse padrão evidencia a presença de economia de escala no consumo energético residencial.

In [4]:
px.histogram(df, x='Household Size', title='Distribuição do Tamanho das Famílias',text_auto=True)

In [5]:
px.bar(
    df.groupby("Household Size", as_index=False)["Energy Consumption (kWh)"].mean(),
    x="Household Size",
    y="Energy Consumption (kWh)",
    title="Consumo Médio (kWh) por Tamanho da Residência",
    text_auto=True
)

In [6]:
px.box(
    df,
    x="Household Size",
    y="Energy Consumption (kWh)",
    color="Household Size",
    title="Distribuição do Consumo por Tamanho da Residência"
)

In [7]:
# corr ≈ +1 → forte relação (mais pessoas → mais consumo)
# corr ≈ 0 → não tem relação clara
# corr < 0 → comportamento estranho (vale investigar)

corr = df["Household Size"].corr(df["Energy Consumption (kWh)"])
print(f"Correlação: {corr:.2f}")

df["kWh_per_person"] = df["Energy Consumption (kWh)"] / df["Household Size"]

px.scatter(
    df,
    x="Household Size",
    y="kWh_per_person",
    color="Appliance Type",
    trendline="ols",
    title="Consumo por Pessoa vs Número de Moradores"
)

Correlação: -0.01


In [8]:
df_grouped = df.groupby(["Household Size", "Appliance Type"], as_index=False)["Energy Consumption (kWh)"].mean()
px.bar(
    df_grouped,
    x="Household Size",
    y="Energy Consumption (kWh)",
    color="Appliance Type",
    barmode="group",
    title="Consumo Médio por Tipo de Aparelho e Tamanho da Casa",
    text_auto=True
)

In [9]:
df["kWh_per_person"] = df["Energy Consumption (kWh)"] / df["Household Size"]

q1 = df["kWh_per_person"].quantile(0.25)
q3 = df["kWh_per_person"].quantile(0.75)
iqr = q3 - q1

outliers = df[df["kWh_per_person"] > (q3 + 1.5 * iqr)]

print(outliers[["Household Size", "Appliance Type", "kWh_per_person"]])

px.box(
    df,
    x="Household Size",
    y="kWh_per_person",
    color="Appliance Type",
    title="Consumo por Pessoa (kWh) por Tipo de Aparelho"
)

       Household Size    Appliance Type  kWh_per_person
3                   1            Heater           3.920
5                   1  Air Conditioning           4.680
14                  1  Air Conditioning           2.880
15                  2            Heater           2.055
23                  2            Heater           2.435
...               ...               ...             ...
99908               1            Heater           3.060
99919               1  Air Conditioning           4.640
99922               1            Heater           3.890
99961               1  Air Conditioning           4.540
99998               1  Air Conditioning           2.690

[6930 rows x 3 columns]


## b) 1 variável quantitativa contínua, na sequência, extraia pelo menos 2 insights da tabela utilizando #. **Energy Consumption (kWh)**

### Esta variável é quantitativa contínua pois representa uma medição física que pode assumir qualquer valor dentro de um intervalo, incluindo casas decimais (ex: 10,55 kWh).
### Insights Extraídos:

- Identificação de Aparelhos Críticos: 

Ao cruzar o consumo com o Appliance Type, podemos identificar quais categorias de aparelhos representam a maior fatia do custo mensal. Se um aparelho específico apresenta um consumo muito acima da média de mercado, Heater e Air Conditioning , isso gera um insight de venda para a GoodWe sugerir a substituição por tecnologias mais eficientes ou integração com baterias.

- Padrões de Desperdício e Falhas: 

Através da análise do consumo contínuo, podemos detectar anomalias. Por exemplo, se o Energy Consumption de uma geladeira permanece alto de forma constante, sem oscilações de termostato, isso é um insight de que o aparelho pode estar com a vedação defeituosa ou motor sobrecarregado, permitindo o envio de alertas de manutenção ao usuário.


In [10]:
px.box(
    df,
    x="Appliance Type",
    y="Energy Consumption (kWh)",
    color="Household Size",
    title="Distribuição do Consumo por Tamanho da Residência"
)

In [11]:
df_grouped = df.groupby(["Appliance Type"], as_index=False)["Energy Consumption (kWh)"].mean()
px.bar(
    df_grouped,
    x="Appliance Type",
    y="Energy Consumption (kWh)",
    barmode="group",
    title="Consumo Médio por Tipo de Aparelho e Tamanho da Casa",
    text_auto=True
)

In [28]:
df['Date'] = pd.to_datetime(df['Date'])

df_resample = (df.set_index('Date').groupby('Appliance Type').resample('D')['Energy Consumption (kWh)'].mean().reset_index())

px.line(
    df_resample,
    x='Date',
    y='Energy Consumption (kWh)',
    color='Appliance Type',
    title='Consumo de Energia ao Longo do Tempo',
    labels={
        'Date': 'Tempo',
        'Energy Consumption (kWh)': 'Consumo (kWh)',
        'Appliance Type': 'Aparelho'
    }
)

In [13]:
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M')
df['hora'] = df['Time'].dt.hour

df_heat = df.groupby(['Appliance Type', 'hora'])['Energy Consumption (kWh)'].mean().reset_index()

fig = px.density_heatmap(
    df_heat,
    x='hora',
    y='Appliance Type',
    z='Energy Consumption (kWh)',
    color_continuous_scale='RdYlBu_r',
    title='Heatmap de Consumo por Hora e Aparelho',
    labels={
        'hora': 'Hora do Dia',
        'Appliance Type': 'Aparelho',
        'Energy Consumption (kWh)': 'Consumo Médio (kWh)'
    },
    text_auto=True
)

fig.show()

In [14]:
px.histogram(
    df,
    x='Energy Consumption (kWh)',
    nbins=30,
    color='Appliance Type',
    title='Distribuição do Consumo de Energia',
    labels={
        'Energy Consumption (kWh)': 'Consumo (kWh)',
        'count': 'Frequência'
    },
    text_auto=True
)

### 03) (2,0 pontos) Elaborar um relatório técnico contendo os principais insights obtidos nas análises realizadas nos itens 01 e 02, destacando de que forma os resultados podem contribuir para a tomada de decisão e/ou geração de valor para a empresa.